# Baseline: Random Forest with HOG Features
### 204466 Deep Learning — Final Project
**Purpose:** Traditional ML baseline to compare with Custom CNN  
**Method:** Extract HOG (Histogram of Oriented Gradients) features → Random Forest classifier

## 1. Setup & Mount Google Drive

In [ ]:
!pip install kaggle scikit-image -q

from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/skin_lesion/'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Save directory: {SAVE_DIR}')

## 2. Download Dataset

In [ ]:
os.environ['KAGGLE_USERNAME'] = 'your_kaggle_username'  # <-- แก้ตรงนี้
os.environ['KAGGLE_KEY']      = 'your_kaggle_api_key'   # <-- แก้ตรงนี้

!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content/ham10000

import zipfile
with zipfile.ZipFile('/content/ham10000/skin-cancer-mnist-ham10000.zip', 'r') as z:
    z.extractall('/content/ham10000')

print('Dataset ready!')

## 3. Import Libraries

In [ ]:
import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from skimage.feature import hog
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)

print('Libraries loaded')

## 4. Load Dataset

In [ ]:
label_map = {
    'nv': 0, 'mel': 1, 'bkl': 2, 'bcc': 3,
    'akiec': 4, 'vasc': 5, 'df': 6
}
class_names = list(label_map.keys())

metadata = pd.read_csv('/content/ham10000/HAM10000_metadata.csv')
metadata['label'] = metadata['dx'].map(label_map)

image_paths = {}
for folder in ['/content/ham10000/HAM10000_images_part_1',
               '/content/ham10000/HAM10000_images_part_2']:
    for img_path in glob.glob(os.path.join(folder, '*.jpg')):
        img_id = os.path.splitext(os.path.basename(img_path))[0]
        image_paths[img_id] = img_path

metadata['path'] = metadata['image_id'].map(image_paths)
metadata = metadata.dropna(subset=['path'])

train_df, temp_df = train_test_split(metadata, test_size=0.30, stratify=metadata['label'], random_state=42)
val_df,   test_df = train_test_split(temp_df,  test_size=0.50, stratify=temp_df['label'],  random_state=42)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

## 5. Extract HOG Features
HOG captures edge and gradient structure — commonly used in traditional CV pipelines.  
Each image is resized to **64×64** and converted to a feature vector.

In [ ]:
def extract_hog_features(df, img_size=64):
    features, labels = [], []
    for _, row in df.iterrows():
        img  = np.array(Image.open(row['path']).convert('RGB').resize((img_size, img_size)))
        feat = hog(img, orientations=9, pixels_per_cell=(8, 8),
                   cells_per_block=(2, 2), channel_axis=-1)
        features.append(feat)
        labels.append(int(row['label']))
    return np.array(features), np.array(labels)

print('Extracting HOG features from train set...')
X_train, y_train = extract_hog_features(train_df)
print(f'Train shape: {X_train.shape}')

print('Extracting HOG features from test set...')
X_test, y_test = extract_hog_features(test_df)
print(f'Test shape : {X_test.shape}')

## 6. Train Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

print('Training Random Forest...')
rf.fit(X_train, y_train)
print('Training complete!')

## 7. Evaluation

In [ ]:
y_pred = rf.predict(X_test)
acc    = accuracy_score(y_test, y_pred)

print(f'Test Accuracy: {acc:.4f} ({acc*100:.2f}%)\n')
print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix — Random Forest')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig(SAVE_DIR + 'rf_confusion_matrix.png', dpi=150)
plt.show()

## 8. Save Results for Comparison

In [ ]:
results = {
    'model'    : 'Random Forest (HOG features)',
    'accuracy' : float(acc),
    'precision': float(precision_score(y_test, y_pred, average='weighted', zero_division=0)),
    'recall'   : float(recall_score(y_test,    y_pred, average='weighted', zero_division=0)),
    'f1'       : float(f1_score(y_test,         y_pred, average='weighted', zero_division=0)),
    'per_class_f1': {
        class_names[i]: float(f)
        for i, f in enumerate(f1_score(y_test, y_pred, average=None, zero_division=0))
    }
}

with open(SAVE_DIR + 'rf_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f'Results saved to {SAVE_DIR}rf_results.json')
print(f'  Accuracy : {results["accuracy"]:.4f}')
print(f'  Precision: {results["precision"]:.4f}')
print(f'  Recall   : {results["recall"]:.4f}')
print(f'  F1       : {results["f1"]:.4f}')